# 🏆 Bonus Track: Edad y género en los ganadores del Óscar

**Pregunta:** ¿Existen diferencias en la edad a la que actores y actrices ganan el Óscar?  
**Hipótesis de partida:** las actrices ganan premios a edades más tempranas que los actores.

**Fuentes:**
- Oscar Awards Dataset (Academy Awards 1927-2023)
- IMDB Name Basics (`name.basics.csv`) — años de nacimiento

---

## 1. Preparación del dataset

### 1.1 Carga y filtrado de nominaciones de interpretación

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import gaussian_kde
from scipy import stats

In [ ]:
oscars = pd.read_csv("data/Oscar_Awards.csv", sep="\t")

# Nos quedamos solo con las categorías de interpretación
acting = oscars[oscars["Class"] == "Acting"]
acting = acting[['Year', 'CanonicalCategory', 'NomId', 'Name', 'NomineeIds', 'Winner']]

print(f"Nominaciones de interpretación: {len(acting):,}")
print("Categorías:", acting["CanonicalCategory"].unique())

### 1.2 Incorporación de años de nacimiento

Se cruza con el dataset de nombres de IMDB (`name.basics.csv`) usando el identificador de nominado como clave.

In [ ]:
names = pd.read_csv("data/name.basics.csv", sep="\t")
names = names[['nconst', 'birthYear']].rename(columns={'nconst': 'NomineeIds'})

acting = acting.merge(names, how='left', on='NomineeIds')

print(f"Registros tras merge: {len(acting):,}")
acting.head()

### 1.3 Limpieza de tipos y valores ausentes

El dataset tiene varios problemas heredados del formato original:
- `Winner` llega como `NaN` para los no ganadores
- `Year` está en string y algunos años tienen formato `"1927/28"`
- `birthYear` es string e incluye `"\N"` para los valores ausentes

In [ ]:
# NaN en Winner = no ganador
acting = acting.fillna(False)

# Normalizar año de ceremonia
acting["Year"] = acting["Year"].str[:4].astype(int)

# Identificar los pocos casos con birthYear ausente
sin_anyo = acting[acting["birthYear"] == "\\N"][['NomineeIds', 'Name']].drop_duplicates()
print(f"Nominados sin año de nacimiento: {len(sin_anyo)}")
print(sin_anyo)

In [ ]:
# Completar manualmente los años de nacimiento ausentes
acting.loc[acting["NomineeIds"] == "nm1503432", "birthYear"] = "1981"  # Catalina Sandino Moreno
acting.loc[acting["NomineeIds"] == "nm0645683", "birthYear"] = "1968"  # Sophie Okonedo
acting.loc[acting["NomineeIds"] == "nm0705152", "birthYear"] = "1948"  # Paul Raci

# Convertir tipos
acting["birthYear"] = acting["birthYear"].astype(int)
acting["Winner"] = acting["Winner"].astype(bool)

acting.dtypes

### 1.4 Variables derivadas: edad y género

- **Edad:** diferencia entre el año de la ceremonia y el año de nacimiento
- **Género:** se infiere de la categoría (`ACTRESS` → F, `ACTOR` → M)

In [ ]:
acting["Age"] = acting["Year"] - acting["birthYear"]
acting["gender"] = acting["CanonicalCategory"].str[:7].apply(lambda x: 'F' if x == 'ACTRESS' else 'M')
acting["decade"] = acting["Year"] // 10 * 10

print(f"Dataset listo: {len(acting):,} nominaciones")
acting[["Name", "Year", "Age", "gender", "Winner"]].sample(5)

---
## 2. Análisis

### 2.1 Distribución de edad en los ganadores

In [ ]:
acting_winners = acting[acting['Winner'] == True]

print("Edad de ganadores por género:")
print(acting_winners.groupby("gender")["Age"].describe().round(1))

In [ ]:
# KDE superpuesto: la forma más clara para comparar distribuciones continuas
hombres = acting_winners[acting_winners['gender'] == 'M']['Age']
mujeres = acting_winners[acting_winners['gender'] == 'F']['Age']

x = np.linspace(0, 100, 200)
kde_m = gaussian_kde(hombres)(x)
kde_f = gaussian_kde(mujeres)(x)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x, y=kde_m, fill='tozeroy', name='Hombres',
    fillcolor='rgba(74, 144, 217, 0.3)',
    line=dict(color='#4A90D9', width=2)
))

fig.add_trace(go.Scatter(
    x=x, y=kde_f, fill='tozeroy', name='Mujeres',
    fillcolor='rgba(245, 166, 35, 0.3)',
    line=dict(color='#F5A623', width=2)
))

fig.update_layout(
    title='Distribución de edad en ganadores del Óscar',
    xaxis_title='Edad', yaxis_title='Densidad',
    template='plotly_dark',
    legend=dict(bgcolor='rgba(0,0,0,0)')
)
fig.show()

La distribución de las mujeres está desplazada claramente hacia la izquierda: ganan a edades más jóvenes y con menos dispersión. Los hombres presentan una distribución más amplia y centrada en torno a los 45-50 años.

### 2.2 Significación estadística

In [ ]:
male_ages = acting_winners[acting_winners['gender'] == 'M']['Age']
female_ages = acting_winners[acting_winners['gender'] == 'F']['Age']

t, p = stats.ttest_ind(male_ages, female_ages)

print(f"Media hombres: {male_ages.mean():.1f} años")
print(f"Media mujeres: {female_ages.mean():.1f} años")
print(f"Diferencia:    {male_ages.mean() - female_ages.mean():.1f} años")
print(f"\nt-test: t={t:.2f}, p={p:.4f}")
print("Diferencia estadisticamente significativa" if p < 0.05 else "No hay diferencia significativa")

La diferencia es estadísticamente significativa (p < 0.05): los actores ganan el Óscar de media casi **10 años más tarde** que las actrices. La hipótesis de partida se confirma.

### 2.3 Evolución temporal: ¿ha cambiado la brecha con el tiempo?

Se calcula la media de edad por década, primero solo para ganadores y luego incluyendo a todos los nominados.

In [ ]:
winner_decade = (
    acting[acting["Winner"] == True]
    .groupby(['decade', 'gender'])["Age"]
    .mean().round(1)
    .reset_index()
)

fig = px.line(
    winner_decade,
    x='decade', y='Age', color='gender',
    markers=True,
    title='Edad media de ganadores del Óscar por década',
    color_discrete_map={'M': '#4A90D9', 'F': '#F5A623'},
    labels={'Age': 'Edad media', 'decade': 'Decada', 'gender': 'Genero'}
)
fig.update_layout(template='plotly_dark')
fig.update_traces(line=dict(width=2), marker=dict(size=8))
fig.show()

Con solo ~4 ganadores por año (40 por década), cualquier caso atípico altera la media considerablemente. Se amplía el análisis incluyendo a todos los nominados para obtener una señal más estable.

In [ ]:
nominees_decade = (
    acting
    .groupby(['decade', 'gender'])["Age"]
    .mean().round(1)
    .reset_index()
)

fig = px.line(
    nominees_decade,
    x='decade', y='Age', color='gender',
    markers=True,
    title='Edad media de nominados al Oscar por decada (ganadores + no ganadores)',
    color_discrete_map={'M': '#4A90D9', 'F': '#F5A623'},
    labels={'Age': 'Edad media', 'decade': 'Decada', 'gender': 'Genero'}
)
fig.update_layout(template='plotly_dark')
fig.update_traces(line=dict(width=2), marker=dict(size=8))
fig.show()

---
## 3. Conclusiones

Con más datos (nominados), el patrón es más claro y consistente:

**La brecha de edad existe y es persistente:**
- Los hombres son nominados y ganan a edades significativamente más altas (~45-50 años de media)
- Las mujeres alcanzan el reconocimiento antes (~35-40 años), con una distribución más concentrada
- La diferencia se mantiene a lo largo de todas las décadas estudiadas, sin una tendencia clara hacia la convergencia

**Evolución:**
- Las actrices ganaban muy jóvenes en los años 20-30 (media de ~25 años) y la tendencia es al alza: cada vez ganan a edades más maduras
- Los actores se han mantenido más estables en el rango 45-50 a lo largo de las décadas

**Limitación del análisis:**  
Solo 4 ganadores por año hacen que la media por década sea sensible a casos extremos. El análisis con todos los nominados ofrece una imagen más robusta y cuenta la misma historia.

| Grupo | Edad media (ganadores) | Rango intercuartílico |
|---|---|---|
| Hombres | ~48 años | ~15 años |
| Mujeres | ~39 años | ~14 años |

> La industria del cine premia a los hombres en la madurez y a las mujeres en su plenitud más temprana — un patrón que los datos reflejan con claridad y que lleva décadas sin cambiar de forma sustancial.